In [1]:
!pip install chronos-forecasting -q
!git clone https://github.com/mlgig/FM4MTSC.git -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.


In [2]:
import numpy as np
import torch
import gc
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from chronos import ChronosPipeline

def get_chronos_embeddings(pipeline, X, batch_size=16):
    all_embeddings = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        batch_flat = batch.reshape(batch.shape[0], -1)
        if batch_flat.shape[1] > 512:
            batch_flat = batch_flat[:, :512]
        context = torch.tensor(batch_flat, dtype=torch.float32)
        with torch.no_grad():
            embeddings, _ = pipeline.embed(context)
        all_embeddings.append(embeddings.float().cpu().numpy().reshape(len(batch), -1))
        del embeddings, context, batch_flat
        torch.cuda.empty_cache()
        gc.collect()
        if i % 500 == 0:
            print(f'  {i}/{len(X)} done')
    return np.concatenate(all_embeddings, axis=0)

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-tiny",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
print('Pipeline ready')

2026-06-15 14:40:36.546149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781534436.728693      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781534436.785936      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781534437.233718      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781534437.233784      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781534437.233787      23 computation_placer.cc:177] computation placer alr

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Pipeline ready


In [3]:
import os
print(os.listdir('/kaggle/input'))

['datasets']


In [4]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/aislingwallace/diss-datasets-2/CMJ.npy
/kaggle/input/datasets/aislingwallace/diss-datasets-2/MP8.npy
/kaggle/input/datasets/aislingwallace/diss-datasets-2/Rowing.npy
/kaggle/input/datasets/aislingwallace/diss-datasets-2/SYNTH.npy
/kaggle/input/datasets/aislingwallace/diss-datasets-2/MP50.npy


In [5]:
name = 'SYNTH'
data = np.load('/kaggle/input/datasets/aislingwallace/diss-datasets-2/SYNTH.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

print('Train embeddings...')
train_emb = get_chronos_embeddings(pipeline, X_tr)
print('Test embeddings...')
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/kaggle/working/Chronos_SYNTH_train_emb.npy', train_emb)
np.save('/kaggle/working/Chronos_SYNTH_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'SYNTH Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

Train embeddings...
  0/7500 done
  2000/7500 done
  4000/7500 done
  6000/7500 done
Test embeddings...
  0/1000 done
SYNTH Accuracy: 0.5680
              precision    recall  f1-score   support

           0       0.57      0.57      0.57       500
           1       0.57      0.57      0.57       500

    accuracy                           0.57      1000
   macro avg       0.57      0.57      0.57      1000
weighted avg       0.57      0.57      0.57      1000



In [6]:
name = 'CMJ'
data = np.load('/kaggle/input/datasets/aislingwallace/diss-datasets-2/CMJ.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

print('Train embeddings...')
train_emb = get_chronos_embeddings(pipeline, X_tr)
print('Test embeddings...')
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/kaggle/working/Chronos_CMJ_train_emb.npy', train_emb)
np.save('/kaggle/working/Chronos_CMJ_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'CMJ Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

Train embeddings...
  0/419 done
Test embeddings...
  0/179 done
CMJ Accuracy: 0.9162
              precision    recall  f1-score   support

           0       0.84      0.97      0.90        60
           1       0.96      0.81      0.88        59
           2       0.97      0.97      0.97        60

    accuracy                           0.92       179
   macro avg       0.92      0.92      0.92       179
weighted avg       0.92      0.92      0.92       179



In [7]:
name = 'MP8'
data = np.load('/kaggle/input/datasets/aislingwallace/diss-datasets-2/MP8.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

print('Train embeddings...')
train_emb = get_chronos_embeddings(pipeline, X_tr)
print('Test embeddings...')
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/kaggle/working/Chronos_MP8_train_emb.npy', train_emb)
np.save('/kaggle/working/Chronos_MP8_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'MP8 Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

Train embeddings...
  0/1426 done
Test embeddings...
  0/595 done
MP8 Accuracy: 0.6050
              precision    recall  f1-score   support

           0       0.63      0.45      0.52       150
           1       0.53      0.69      0.60       140
           2       0.53      0.61      0.57       150
           3       0.79      0.67      0.73       155

    accuracy                           0.61       595
   macro avg       0.62      0.61      0.60       595
weighted avg       0.62      0.61      0.61       595



In [8]:
name = 'MP50'
data = np.load('/kaggle/input/datasets/aislingwallace/diss-datasets-2/MP50.npy', allow_pickle=True).item()
X_tr, y_tr = data['train']['X'], np.array([int(x) for x in data['train']['y']])
X_te, y_te = data['test']['X'], np.array([int(x) for x in data['test']['y']])

print('Train embeddings...')
train_emb = get_chronos_embeddings(pipeline, X_tr)
print('Test embeddings...')
test_emb = get_chronos_embeddings(pipeline, X_te)
np.save('/kaggle/working/Chronos_MP50_train_emb.npy', train_emb)
np.save('/kaggle/working/Chronos_MP50_test_emb.npy', test_emb)

clf = RidgeClassifier()
clf.fit(train_emb, y_tr)
y_pred = clf.predict(test_emb)
print(f'MP50 Accuracy: {accuracy_score(y_te, y_pred):.4f}')
print(classification_report(y_te, y_pred))
del train_emb, test_emb, X_tr, X_te, data
gc.collect()
torch.cuda.empty_cache()

Train embeddings...
  0/1426 done
Test embeddings...
  0/595 done
MP50 Accuracy: 0.4689
              precision    recall  f1-score   support

           0       0.47      0.29      0.36       150
           1       0.35      0.39      0.37       140
           2       0.42      0.46      0.44       150
           3       0.63      0.72      0.67       155

    accuracy                           0.47       595
   macro avg       0.46      0.47      0.46       595
weighted avg       0.47      0.47      0.46       595

